# Project 5 - Code Breakers

In this project, I work with a simple text encryption method based on ASCII codes.  
The main idea is that every character can be converted into a number using `ord()`, and every number can be converted back into a character using `chr()`.

The encrypted file contains a sequence of numbers. My goal is to find the secret key and recover the original English text.

Before trying to break the encrypted file, I first build and test small helper functions. This helps me understand the encryption rule step by step.

In [17]:
def str_to_ascii(s):
    # This list will store the ASCII values of each character in the string.
    ascii_codes = []

    # We go through the string one character at a time.
    # For example, if s = "Hi", we first see 'H', then 'i'.
    for c in s:

        # ord(c) converts a character into its ASCII number.
        # For example:
        # 'H' -> 72
        # 'i' -> 105
        ascii_value = ord(c)

        # We store this number in our list.
        ascii_codes.append(ascii_value)

    # At the end, we return the full list of ASCII values.
    return ascii_codes

In [18]:
def ascii_to_str(ascii_codes):
    # This list will store characters after converting from ASCII.
    char_list = []

    # We go through each ASCII number in the list.
    for n in ascii_codes:

        # chr(n) converts an ASCII number back into a character.
        # For example:
        # 72 -> 'H'
        # 105 -> 'i'
        char = chr(n)

        # Store the character in the list.
        char_list.append(char)

    # Join all characters together into one string.
    # ['H', 'i'] -> "Hi"
    return ''.join(char_list)

To make sure these two functions work correctly, I test them on a simple sentence.  
If I convert the sentence into ASCII codes and then convert it back, I should get the same sentence again.

In [19]:
test_message = "This is MTH 337!"

ascii_test = str_to_ascii(test_message)
new_message = ascii_to_str(ascii_test)

print(ascii_test)
print(new_message)

[84, 104, 105, 115, 32, 105, 115, 32, 77, 84, 72, 32, 51, 51, 55, 33]
This is MTH 337!


Now I need to handle the secret key.

The key may be shorter than the message. Since the encryption rule works character by character, the key must have the same length as the message.  
So I repeat the key again and again until it is long enough, and then I cut it to the exact length I need.

In [20]:
def get_padded_key_ascii(key_ascii, length):
    # This list will store the repeated key values.
    padded_key = []

    # We keep adding elements until the padded key reaches the desired length.
    # The length we want is the same as the message.
    while len(padded_key) < length:

        # Go through each value in the original key.
        for n in key_ascii:

            # Only add more values if we still need them.
            # This avoids going over the target length.
            if len(padded_key) < length:
                padded_key.append(n)

    # Now padded_key has exactly the required length.
    return padded_key

In [21]:
# Test the padded key function.
key = "cat"
key_ascii = str_to_ascii(key)

padded_key = get_padded_key_ascii(key_ascii, 10)

print(key_ascii)
print(padded_key)
print(len(padded_key))

[99, 97, 116]
[99, 97, 116, 99, 97, 116, 99, 97, 116, 99]
10


Next I write the encryption function.

The encryption rule says that each message ASCII value and key ASCII value are added together.  
Then I take the remainder after division by 128. This keeps the result inside the ASCII range.

In [22]:
def encrypt(message_ascii, key_ascii):
    # This list will store the encrypted ASCII values.
    encrypted_ascii = []

    # First, we make the key the same length as the message.
    # This is necessary because encryption is done position by position.
    padded_key_ascii = get_padded_key_ascii(key_ascii, len(message_ascii))

    # Now we go through the message one position at a time.
    for i in range(len(message_ascii)):

        # message_ascii[i] is the original ASCII value (m_i)
        # padded_key_ascii[i] is the key ASCII value (k_i)

        # According to the encryption rule:
        # c_i = (m_i + k_i) % 128
        total = message_ascii[i] + padded_key_ascii[i]

        # We use modulo 128 to keep the result within valid ASCII range.
        encrypted_value = total % 128

        # Store the encrypted value.
        encrypted_ascii.append(encrypted_value)

    return encrypted_ascii

Decryption works backwards.

During encryption, the key value was added.  
So during decryption, I subtract the key value.  
I still use % 128 because the result may become negative, and modulo brings it back into the ASCII range.

In [23]:
def decrypt(encrypted_ascii, key_ascii):
    # This list will store the decrypted ASCII values.
    decrypted_ascii = []

    # Make the key the same length as the encrypted message.
    padded_key_ascii = get_padded_key_ascii(key_ascii, len(encrypted_ascii))

    # Go through each encrypted number.
    for i in range(len(encrypted_ascii)):

        # encrypted_ascii[i] is c_i
        # padded_key_ascii[i] is k_i

        # Reverse the encryption:
        # m_i = c_i - k_i
        difference = encrypted_ascii[i] - padded_key_ascii[i]

        # Use modulo 128 to keep the value in ASCII range.
        decrypted_value = difference % 128

        # Store the result.
        decrypted_ascii.append(decrypted_value)

    # Convert ASCII back to string.
    return ascii_to_str(decrypted_ascii)

Before using these functions on the real encrypted file, I test them using my own message and key.  
If everything is correct, encrypting and then decrypting should give back the original message.

In [24]:
message = "Top secret!"
key = "buffalo"

message_ascii = str_to_ascii(message)
key_ascii = str_to_ascii(key)

encrypted = encrypt(message_ascii, key_ascii)
decrypted = decrypt(encrypted, key_ascii)

print("Original message:")
print(message)

print("\nEncrypted ASCII:")
print(encrypted)

print("\nDecrypted message:")
print(decrypted)

Original message:
Top secret!

Encrypted ASCII:
[54, 100, 86, 6, 84, 81, 82, 84, 90, 90, 7]

Decrypted message:
Top secret!


The test works if the decrypted message is the same as the original message.  
Now I can use the same decryption function to attack the encrypted file.

Since the project says the secret key is a word from the dictionary, I can try every word in the dictionary as a possible key.

In [25]:
# Read the encrypted file.
# Change this filename to the file assigned to you.
filename = "xsun22.txt"

with open(filename, "r") as file:
    encrypted_text = file.read()

print(encrypted_text[:200])

64 92 1 95 84 23 75 73 77 83 90 77 69 24 5 107 80 66 96 5 88 78 85 81 87 23 92 73 81 5 71 73 83 95 74 92 15 84 12 73 96 91 66 92 85 92 73 83 77 83 90 77 13 12 46 23 95 66 95 5 95 77 1 99 77 102 85 107


The encrypted file should contain numbers separated by commas.  
I need to convert the file content from text into a list of integers.

In [26]:
# Split the text into parts
# We use .split() instead of .split(",") because the file is space-separated,
# not comma-separated.
encrypted_parts = encrypted_text.split()

encrypted_ascii = []

# Go through each piece of text
for part in encrypted_parts:

    # Remove any extra spaces or newline characters
    part = part.strip()

    # Only convert non-empty parts
    if part != "":
        # Convert the string into an integer
        encrypted_ascii.append(int(part))

# Print first 20 values to check correctness
print(encrypted_ascii[:20])

# Print total length (useful for debugging)
print("Length of encrypted message:", len(encrypted_ascii))

[64, 92, 1, 95, 84, 23, 75, 73, 77, 83, 90, 77, 69, 24, 5, 107, 80, 66, 96, 5]
Length of encrypted message: 1360


Now I read the dictionary file.  
Each line contains one possible English word.  
I remove extra spaces and line breaks so that each word can be used as a possible key.

In [27]:
with open("dictionary.txt", "r") as file:
    dictionary_words = file.readlines()

clean_words = []

for word in dictionary_words:
    word = word.strip()

    if word != "":
        clean_words.append(word)

print("Number of words in dictionary:", len(clean_words))
print(clean_words[:10])

Number of words in dictionary: 61406
['A', 'a', 'Aachen', 'Aalborg', 'aardvark', 'Aarhus', 'Aaron', 'AB', 'Ab', 'abaci']


If I try every word as a key, I need a way to decide whether the decrypted result looks like English.

A simple way is to count common English words such as "the", "and", "of", "to", and "in".  
A correct decrypted message should usually contain many common English words.

In [28]:
def english_score(text):
    # Convert the text to lowercase so matching is easier.
    text_lower = text.lower()

    # These are very common English words.
    # A real English paragraph usually contains many of them.
    common_words = [
        " the ", " and ", " of ", " to ", " in ",
        " that ", " is ", " it ", " for ", " was "
    ]

    score = 0

    # Count how many times each common word appears.
    for word in common_words:
        count = text_lower.count(word)

        # Add to score.
        score = score + count

    return score

Now I try every dictionary word as a possible key.

For each word:
1. I convert the word into ASCII codes.
2. I decrypt the encrypted message using that word as the key.
3. I calculate an English score.
4. I keep the best result.

In [29]:
best_score = -1
best_key = ""
best_message = ""

# Try every word in the dictionary as a possible key.
for word in clean_words:

    # Convert the word into ASCII values.
    key_ascii = str_to_ascii(word)

    # Attempt to decrypt the message using this key.
    possible_message = decrypt(encrypted_ascii, key_ascii)

    # Evaluate how "English-like" the result is.
    score = english_score(possible_message)

    # If this result is better than all previous ones, save it.
    if score > best_score:
        best_score = score
        best_key = word
        best_message = possible_message


# Print results.
print("Best key:", best_key)
print("\nScore:", best_score)
print("\nDecrypted message preview:")
print(best_message[:500])

Best key: whale

Score: 46

Decrypted message preview:
It so chanced, that after the Parsee's disappearance, I was he whom
the Fates ordained to take the place of Ahab's bowsman, when that
bowsman assumed the vacant post; the same, who, when on the last day the
three men were tossed from out of the rocking boat, was dropped astern.
So, floating on the margin of the ensuing scene, and in full sight of
it, when the halfspent suction of the sunk ship reached me, I was then,
but slowly, drawn towards the closing vortex. When I reached it, it had
subsid


The output above gives the key with the highest English score.  
To make sure it is really correct, I print more of the decrypted message and check whether it reads like normal English.

In [30]:
print(best_message)

It so chanced, that after the Parsee's disappearance, I was he whom
the Fates ordained to take the place of Ahab's bowsman, when that
bowsman assumed the vacant post; the same, who, when on the last day the
three men were tossed from out of the rocking boat, was dropped astern.
So, floating on the margin of the ensuing scene, and in full sight of
it, when the halfspent suction of the sunk ship reached me, I was then,
but slowly, drawn towards the closing vortex. When I reached it, it had
subsided to a creamy pool. Round and round, then, and ever contracting
towards the button-like black bubble at the axis of that slowly wheeling
circle, like another Ixion I did revolve. Till, gaining that vital
centre, the black bubble upward bur st; and now, liberated by reason of
its cunning spring, and, owing to its great buoyancy, rising with great
force, the coffin life-buoy shot lengthwise from the sea, fell over, and
floated by my side. Buoyed up by that coffin, for almo st one whole day
and n

## Conclusion

In this project, I used ASCII codes to understand and reverse a simple encryption method.  
First, I wrote helper functions to convert between strings and ASCII codes. Then I wrote functions for padding the key, encrypting a message, and decrypting a message.

After testing the functions with my own example, I used the dictionary file to try many possible keys.  
For each possible key, I decrypted the encrypted text and gave the result an English score based on common English words.  
The key with the highest score produced readable English text, so I used it as the secret key.